# 25 Local Post-to-Trend Matching

Deterministic local-only staged matching from Bluesky topic candidates to normalized Twitter trends.

- No Snowflake
- No reruns of collection/enrichment
- No ML training in this phase


**Notebook purpose:** Runs staged (exact/fuzzy/semantic) matching from Bluesky topic candidates to normalized Twitter trends. Produces full-match and best-match parquet outputs.

**Required data:** `local/derived/bluesky/bluesky_topic_candidates.parquet` (from notebook 24) and `local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet` (from notebook 22).

**Run order:** Run after notebook 24 (topic extraction). Run before notebook 26 (feature engineering).

## 1. Load Inputs and Inspect Schemas


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'src' / 'nlp' / 'post_trend_matching.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing src/nlp/post_trend_matching.py')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nlp.post_trend_matching import MatchConfig, match_post_candidates_to_trends

candidates_path = ROOT / "local/derived/bluesky/bluesky_topic_candidates.parquet"
trends_path = ROOT / "local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet"

for _p, _label in [(candidates_path, "notebook 24 (topic candidates)"), (trends_path, "notebook 22 (trend normalization)")]:
    if not _p.exists():
        raise FileNotFoundError(f"DATA NOT YET AVAILABLE -- run {_label} first.\nMissing: {_p}")

candidates_df = pd.read_parquet(candidates_path)
trends_df = pd.read_parquet(trends_path)

print("Candidate rows:", len(candidates_df), "columns:", len(candidates_df.columns))
print("Trend rows:", len(trends_df), "columns:", len(trends_df.columns))
print("Candidate columns:", candidates_df.columns.tolist())
print("Trend columns:", trends_df.columns.tolist())

Candidate rows: 237351 columns: 17
Trend rows: 101731 columns: 20
Candidate columns: ['uri', 'post_created_at', 'post_text_raw', 'post_text_clean', 'post_text_alnum', 'candidate_phrase_raw', 'candidate_phrase_clean', 'candidate_phrase_alnum', 'candidate_phrase_no_hash', 'candidate_source_type', 'candidate_token_count', 'candidate_char_count', 'candidate_rank_in_post', 'is_hashtag_candidate', 'contains_digit', 'is_unigram_fallback', 'candidate_start_index']
Trend columns: ['num_hours', 'date', 'name', 'counts', 'trend_name_raw', 'trend_name_clean', 'trend_name_clean_no_hash', 'trend_name_clean_no_dollar', 'trend_name_alnum', 'trend_name_token_count', 'trend_name_char_count', 'is_blank_raw', 'is_hashtag', 'has_special_chars', 'has_non_ascii', 'has_url_like', 'normalized_key_with_hash', 'normalized_key_no_hash', 'normalized_key_no_dollar', 'normalized_date']


## 2. Configure and Run Staged Matching


In [2]:
cfg = MatchConfig(
    date_window_days=3,
    fuzzy_min_score=0.88,
    semantic_min_score=0.60,
    max_stage_pool=5000,
    semantic_pool_top_k=250,
)
cfg


MatchConfig(date_window_days=3, fuzzy_min_score=0.88, semantic_min_score=0.6, max_stage_pool=5000, semantic_pool_top_k=250)

In [3]:
full_matches_df, best_matches_df, summary = match_post_candidates_to_trends(
    candidates_df=candidates_df,
    trends_df=trends_df,
    config=cfg,
)

full_matches_df = full_matches_df.sort_values(["candidate_id", "stage_rank", "trend_name_clean"], kind="stable").reset_index(drop=True)
best_matches_df = best_matches_df.sort_values(["uri"], kind="stable").reset_index(drop=True)

print("Full match rows:", len(full_matches_df))
print("Best match rows:", len(best_matches_df))
summary


  Building TF-IDF index...
  Building FAISS semantic index...


/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15241.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 398/398 [00:07<00:00, 52.36it/s]


  Index build complete.
  Matching candidate 0 / 237,351 (0%)
  Matching candidate 11,867 / 237,351 (4%)
  Matching candidate 23,734 / 237,351 (9%)
  Matching candidate 35,601 / 237,351 (14%)
  Matching candidate 47,468 / 237,351 (19%)
  Matching candidate 59,335 / 237,351 (24%)
  Matching candidate 71,202 / 237,351 (29%)
  Matching candidate 83,069 / 237,351 (34%)
  Matching candidate 94,936 / 237,351 (39%)
  Matching candidate 106,803 / 237,351 (44%)
  Matching candidate 118,670 / 237,351 (49%)
  Matching candidate 130,537 / 237,351 (54%)
  Matching candidate 142,404 / 237,351 (59%)
  Matching candidate 154,271 / 237,351 (64%)
  Matching candidate 166,138 / 237,351 (69%)
  Matching candidate 178,005 / 237,351 (74%)
  Matching candidate 189,872 / 237,351 (79%)
  Matching candidate 201,739 / 237,351 (84%)
  Matching candidate 213,606 / 237,351 (89%)
  Matching candidate 225,473 / 237,351 (94%)
  Matching candidate 237,340 / 237,351 (99%)
  Matching complete: 237,351 candidates processe

{'config': {'date_window_days': 3,
  'fuzzy_min_score': 0.88,
  'semantic_min_score': 0.6,
  'max_stage_pool': 5000,
  'semantic_pool_top_k': 250},
 'total_candidate_rows': 301761,
 'total_unique_candidates': 237351,
 'total_unique_posts': 17958,
 'candidate_stage_counts': {'unmatched': 224092,
  'semantic': 10759,
  'exact': 1350,
  'fuzzy': 1150},
 'post_stage_counts': {'unmatched': 11554,
  'semantic': 4447,
  'exact': 1134,
  'fuzzy': 823},
 'post_matched_rate': 0.3566098674685377,
 'ambiguous_candidate_count': 8790,
 'ambiguous_candidate_rate': 0.037033760127406244,
 'full_row_stage_counts': {'unmatched': 224092,
  'semantic': 62784,
  'exact': 9664,
  'fuzzy': 5221},
 'temporal_pool_mode_counts': {'global_fallback_no_date_overlap': 226095,
  'global_fallback_no_candidate_date': 11256},
 'top_exact_trends': [{'trend_name_clean': 'pam bondi', 'count': 936},
  {'trend_name_clean': 'good friday', 'count': 660},
  {'trend_name_clean': 'iran', 'count': 429},
  {'trend_name_clean': 'nas

## 3. Stage-by-Stage Outcomes


In [4]:
stage_row_counts = full_matches_df["match_stage"].value_counts(dropna=False)
stage_candidate_counts = (
    full_matches_df.sort_values(["candidate_id", "stage_rank"], kind="stable")
    .groupby("candidate_id", as_index=False)
    .head(1)["match_stage"]
    .value_counts(dropna=False)
)
temporal_pool_counts = (
    full_matches_df.sort_values(["candidate_id", "stage_rank"], kind="stable")
    .groupby("candidate_id", as_index=False)
    .head(1)["temporal_pool_mode"]
    .value_counts(dropna=False)
)

print("Stage row counts:")
print(stage_row_counts)
print()
print("Stage candidate counts:")
print(stage_candidate_counts)
print()
print("Temporal pool counts:")
print(temporal_pool_counts)


Stage row counts:
match_stage
unmatched    224092
semantic      62784
exact          9664
fuzzy          5221
Name: count, dtype: int64

Stage candidate counts:
match_stage
unmatched    224092
semantic      10759
exact          1350
fuzzy          1150
Name: count, dtype: int64

Temporal pool counts:
temporal_pool_mode
global_fallback_no_date_overlap      226095
global_fallback_no_candidate_date     11256
Name: count, dtype: int64


## 4. Representative Examples

Includes exact, fuzzy, semantic, ambiguous, and unmatched examples.


In [5]:
exact_examples = full_matches_df.loc[full_matches_df["match_stage"] == "exact", [
    "uri", "candidate_phrase_alnum", "trend_name_clean", "match_score", "trend_counts", "temporal_pool_mode"
]].head(10)
fuzzy_examples = full_matches_df.loc[full_matches_df["match_stage"] == "fuzzy", [
    "uri", "candidate_phrase_alnum", "trend_name_clean", "match_score", "trend_counts", "temporal_pool_mode"
]].head(10)
semantic_examples = full_matches_df.loc[full_matches_df["match_stage"] == "semantic", [
    "uri", "candidate_phrase_alnum", "trend_name_clean", "match_score", "semantic_token_cosine", "semantic_char_trigram_jaccard", "temporal_pool_mode"
]].head(10)
ambiguous_examples = full_matches_df.loc[full_matches_df["is_ambiguous"], [
    "candidate_id", "uri", "candidate_phrase_alnum", "trend_name_clean", "match_stage", "match_score", "ambiguity_count"
]].head(20)
unmatched_examples = full_matches_df.loc[full_matches_df["match_stage"] == "unmatched", [
    "uri", "candidate_phrase_alnum", "temporal_pool_mode"
]].head(20)

print("Exact examples")
exact_examples


Exact examples


,uri,candidate_phrase_alnum,trend_name_clean,match_score,trend_counts,temporal_pool_mode
104,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,nurse,nurse,1.0,40038.0,global_fallback_no_date_overlap
105,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,nurse,nurse,1.0,32486.0,global_fallback_no_date_overlap
167,at://did:plc:22l2z4p4qhhyobx22dfcvijs/app.bsky...,dragon,dragon,1.0,136659.0,global_fallback_no_date_overlap
168,at://did:plc:22l2z4p4qhhyobx22dfcvijs/app.bsky...,dragon,dragon,1.0,131674.0,global_fallback_no_date_overlap
169,at://did:plc:22l2z4p4qhhyobx22dfcvijs/app.bsky...,dragon,dragon,1.0,91614.0,global_fallback_no_date_overlap
170,at://did:plc:22l2z4p4qhhyobx22dfcvijs/app.bsky...,dragon,dragon,1.0,90655.0,global_fallback_no_date_overlap
172,at://did:plc:22l2z4p4qhhyobx22dfcvijs/app.bsky...,shrek,shrek,1.0,646917.0,global_fallback_no_date_overlap
173,at://did:plc:22l2z4p4qhhyobx22dfcvijs/app.bsky...,shrek,shrek,1.0,445486.0,global_fallback_no_date_overlap
174,at://did:plc:22l2z4p4qhhyobx22dfcvijs/app.bsky...,shrek,shrek,1.0,212174.0,global_fallback_no_date_overlap
175,at://did:plc:22l2z4p4qhhyobx22dfcvijs/app.bsky...,shrek,shrek,1.0,22178.0,global_fallback_no_date_overlap


In [6]:
print("Fuzzy examples")
fuzzy_examples


Fuzzy examples


,uri,candidate_phrase_alnum,trend_name_clean,match_score,trend_counts,temporal_pool_mode
60,at://did:plc:22auflhdbiemhxvkgvpusmmd/app.bsky...,does something,do something,0.923077,159205.0,global_fallback_no_date_overlap
61,at://did:plc:22auflhdbiemhxvkgvpusmmd/app.bsky...,does something,do something,0.923077,150778.0,global_fallback_no_date_overlap
290,at://did:plc:22w4xuatznlr2a65xslrqebv/app.bsky...,crimes against humanity,cards against humanity,0.888889,5700.0,global_fallback_no_date_overlap
291,at://did:plc:22w4xuatznlr2a65xslrqebv/app.bsky...,crimes against humanity,cards against humanity,0.888889,5070.0,global_fallback_no_date_overlap
767,at://did:plc:24w74q2zfkzemkikzweooohi/app.bsky...,saintjohn,saint john,0.947368,1153.0,global_fallback_no_date_overlap
918,at://did:plc:25em24uhdhjzsats4pugqlf5/app.bsky...,happy ending,happy opening,0.880000,16098.0,global_fallback_no_date_overlap
919,at://did:plc:25em24uhdhjzsats4pugqlf5/app.bsky...,happy ending,happy opening,0.880000,13436.0,global_fallback_no_date_overlap
2213,at://did:plc:2cu6p3d5aylzwyx3o4czivry/app.bsky...,dear leader,dear reader,0.909091,35740.0,global_fallback_no_date_overlap
2434,at://did:plc:2dkt6jd7o3apvp5edpsn7gzz/app.bsky...,suomi,somi,0.888889,7446.0,global_fallback_no_candidate_date
2631,at://did:plc:2elx44ewsny25y7o6nkdgs7h/app.bsky...,gold stars,gold star,0.947368,415297.0,global_fallback_no_date_overlap


In [7]:
print("Semantic examples")
semantic_examples


Semantic examples


,uri,candidate_phrase_alnum,trend_name_clean,match_score,semantic_token_cosine,semantic_char_trigram_jaccard,temporal_pool_mode
6,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,cycle about the president,the president,0.638453,0.707107,0.478261,global_fallback_no_date_overlap
23,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,expected the dow,the dow,0.678690,0.816497,0.357143,global_fallback_no_date_overlap
24,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,expected the dow,the dow,0.678690,0.816497,0.357143,global_fallback_no_date_overlap
25,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,expected the dow,the dow,0.678690,0.816497,0.357143,global_fallback_no_date_overlap
26,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,expected the dow,the dow,0.678690,0.816497,0.357143,global_fallback_no_date_overlap
46,at://did:plc:223eaz6uecpa3t63jdlojutp/app.bsky...,done wishing,wishing,0.644975,0.707107,0.500000,global_fallback_no_date_overlap
47,at://did:plc:223eaz6uecpa3t63jdlojutp/app.bsky...,done wishing,wishing,0.644975,0.707107,0.500000,global_fallback_no_date_overlap
95,at://did:plc:22bixok3zcw6dv72gyi5pwox/app.bsky...,ain t no,ain t nobody,0.646667,0.666667,0.600000,global_fallback_no_date_overlap
103,at://did:plc:22bixok3zcw6dv72gyi5pwox/app.bsky...,being a dick,being a dad,0.641667,0.666667,0.583333,global_fallback_no_date_overlap
141,at://did:plc:22g76b5uv7d7gpsriovic5sy/app.bsky...,forever and ever,forever and always,0.633333,0.666667,0.555556,global_fallback_no_date_overlap


In [8]:
print("Ambiguous examples")
ambiguous_examples


Ambiguous examples


,candidate_id,uri,candidate_phrase_alnum,trend_name_clean,match_stage,match_score,ambiguity_count
23,23,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,expected the dow,the dow,semantic,0.678690,4
24,23,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,expected the dow,the dow,semantic,0.678690,4
25,23,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,expected the dow,the dow,semantic,0.678690,4
26,23,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,expected the dow,the dow,semantic,0.678690,4
46,43,at://did:plc:223eaz6uecpa3t63jdlojutp/app.bsky...,done wishing,wishing,semantic,0.644975,2
47,43,at://did:plc:223eaz6uecpa3t63jdlojutp/app.bsky...,done wishing,wishing,semantic,0.644975,2
60,56,at://did:plc:22auflhdbiemhxvkgvpusmmd/app.bsky...,does something,do something,fuzzy,0.923077,2
61,56,at://did:plc:22auflhdbiemhxvkgvpusmmd/app.bsky...,does something,do something,fuzzy,0.923077,2
104,99,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,nurse,nurse,exact,1.000000,2
105,99,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,nurse,nurse,exact,1.000000,2


In [9]:
print("Unmatched examples")
unmatched_examples


Unmatched examples


,uri,candidate_phrase_alnum,temporal_pool_mode
0,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,does finally die there,global_fallback_no_date_overlap
1,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,finally die there ll,global_fallback_no_date_overlap
2,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,die there ll arise,global_fallback_no_date_overlap
3,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,ll arise a new,global_fallback_no_date_overlap
4,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,arise a new myth,global_fallback_no_date_overlap
5,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,new myth cycle about,global_fallback_no_date_overlap
7,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,president who waits resting,global_fallback_no_date_overlap
8,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,resting in his tomb,global_fallback_no_date_overlap
9,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,until when the need,global_fallback_no_date_overlap
10,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,need is most dire,global_fallback_no_date_overlap


## 5. Write Required Outputs


In [10]:
matching_dir = ROOT / "local/derived/matching"
sample_dir = ROOT / "data/samples"
matching_dir.mkdir(parents=True, exist_ok=True)
sample_dir.mkdir(parents=True, exist_ok=True)

full_out = matching_dir / "bluesky_post_trend_matches.parquet"
best_out = matching_dir / "bluesky_post_best_trend_matches.parquet"
sample_parquet_out = sample_dir / "bluesky_post_trend_matches_sample_1000.parquet"
sample_csv_out = sample_dir / "bluesky_post_trend_matches_sample_1000.csv"
summary_out = matching_dir / "bluesky_post_trend_matching_summary.json"

full_matches_df.to_parquet(full_out, index=False)
best_matches_df.to_parquet(best_out, index=False)
full_matches_df.head(1000).to_parquet(sample_parquet_out, index=False)
full_matches_df.head(1000).to_csv(sample_csv_out, index=False)

summary_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "25_local_post_to_trend_matching",
    "config": cfg.__dict__,
    "input_paths": {"candidates": str(candidates_path), "trends": str(trends_path)},
    "summary": summary,
    "stage_row_counts": {str(k): int(v) for k, v in stage_row_counts.items()},
    "stage_candidate_counts": {str(k): int(v) for k, v in stage_candidate_counts.items()},
    "temporal_pool_mode_counts": {str(k): int(v) for k, v in temporal_pool_counts.items()},
    "output_paths": {
        "full_matches_parquet": str(full_out),
        "best_matches_parquet": str(best_out),
        "sample_matches_parquet": str(sample_parquet_out),
        "sample_matches_csv": str(sample_csv_out),
    },
}
summary_out.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("Wrote:", full_out)
print("Wrote:", best_out)
print("Wrote:", sample_parquet_out)
print("Wrote:", sample_csv_out)
print("Wrote:", summary_out)

Wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/matching/bluesky_post_trend_matches.parquet
Wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/matching/bluesky_post_best_trend_matches.parquet
Wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/bluesky_post_trend_matches_sample_1000.parquet
Wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/bluesky_post_trend_matches_sample_1000.csv
Wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/matching/bluesky_post_trend_matching_summary.json
